## Configurazione

In [ ]:
import os
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import itertools
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
import glob


#PATH per i relativi dataset
MVTEC_DIR = '/content/drive/MyDrive/Project Work CV/MVTec1'


#Categorie da codificare
LABELS = ['leather', 'pill', 'tile', 'transistor', 'wood']


# Dataset

## Caricamento dataset MVTec da google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
class RGB_to_BGR(object):
    def __call__(self, tensor):
        return tensor[[2,1,0], ...]

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    RGB_to_BGR()
])

## Dataset di test

In [ ]:
#Uso una classe perché non posso usare ImageFolder
#ImgaeFolder non è in grado di navigare nelle sottocartelle di test
class MVTecTestDataset(Dataset):
    def __init__(self, root_dir, category, transform):
        self.transform = transform
        #tutti i path delle immagini in test
        self.image_paths = []

        #percorso della cartella di test per la specifica categoria
        test_dir = os.path.join(root_dir, category, 'test')

        self.image_paths = glob.glob(os.path.join(test_dir, "*", "*.png"))

    #per il dataloader
    def __len__(self):
        return len(self.image_paths)

    #funzione per ottenere l'immagine
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        #prendo tutte le sotto cartelle di test (tipi di anomalia)
        subfolders = img_path.split(os.sep)
        img_type = subfolders[-2]
        #nome immagine
        img_name = subfolders[-1]
        #nome categoria
        category = subfolders[-4]

        #applicazione transform all'immagine per resize e BGR
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        #se l'immagine non ha anomalia --> label = 0
        #se l'immagine ha un qualsiasi tipo di anomalia --> label 1
        if img_type == 'good':
            label = 0
            mask = torch.zeros((1, image.shape[1], image.shape[2]))
        else:
            label = 1 
            #prendo la maschera ground_truth dell'immagine corrispondente
            root_path = os.sep.join(subfolders[:-4])
            mask_filename = img_name.replace(".png", "_mask.png")
            mask_path = os.path.join(root_path, category, 'ground_truth', img_type, mask_filename)

            #converto la maschera in scala di grigi
            mask = Image.open(mask_path).convert('L')

            
            mask_transform = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.ToTensor()
            ])
            mask = mask_transform(mask)

        return image, mask, label

# Modello DRAEM - UNET

In [ ]:
import torch
import torch.nn as nn


class ReconstructiveSubNetwork(nn.Module):
    def __init__(self,in_channels=3, out_channels=3, base_width=128):
        super(ReconstructiveSubNetwork, self).__init__()
        self.encoder = EncoderReconstructive(in_channels, base_width)
        self.decoder = DecoderReconstructive(base_width, out_channels=out_channels)

    def forward(self, x):
        b5 = self.encoder(x)
        output = self.decoder(b5)
        return output

    def getName(self):
        return self.__class__.__name__


class EncoderReconstructive(nn.Module):
    def __init__(self, in_channels, base_width):
        super(EncoderReconstructive, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels,base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True))
        self.mp1 = nn.Sequential(nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(base_width,base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True))
        self.mp2 = nn.Sequential(nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(base_width*2,base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True))
        self.mp3 = nn.Sequential(nn.MaxPool2d(2))
        self.block4 = nn.Sequential(
            nn.Conv2d(base_width*4,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))
        self.mp4 = nn.Sequential(nn.MaxPool2d(2))
        self.block5 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))


    def forward(self, x):
        b1 = self.block1(x)
        mp1 = self.mp1(b1)
        b2 = self.block2(mp1)
        mp2 = self.mp3(b2)
        b3 = self.block3(mp2)
        mp3 = self.mp3(b3)
        b4 = self.block4(mp3)
        mp4 = self.mp4(b4)
        b5 = self.block5(mp4)
        return b5


class DecoderReconstructive(nn.Module):
    def __init__(self, base_width, out_channels=1):
        super(DecoderReconstructive, self).__init__()

        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 8, base_width * 8, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 8),
                                 nn.ReLU(inplace=True))
        self.db1 = nn.Sequential(
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 8, base_width * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 4),
            nn.ReLU(inplace=True)
        )

        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 4, base_width * 4, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 4),
                                 nn.ReLU(inplace=True))
        self.db2 = nn.Sequential(
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 4, base_width * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 2),
            nn.ReLU(inplace=True)
        )

        self.up3 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 2, base_width*2, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width*2),
                                 nn.ReLU(inplace=True))
        # cat with base*1
        self.db3 = nn.Sequential(
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*1, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*1),
            nn.ReLU(inplace=True)
        )

        self.up4 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width),
                                 nn.ReLU(inplace=True))
        self.db4 = nn.Sequential(
            nn.Conv2d(base_width*1, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True)
        )

        self.fin_out = nn.Sequential(nn.Conv2d(base_width, out_channels, kernel_size=3, padding=1))
        #self.fin_out = nn.Conv2d(base_width, out_channels, kernel_size=3, padding=1)

    def forward(self, b5):
        up1 = self.up1(b5)
        db1 = self.db1(up1)

        up2 = self.up2(db1)
        db2 = self.db2(up2)

        up3 = self.up3(db2)
        db3 = self.db3(up3)

        up4 = self.up4(db3)
        db4 = self.db4(up4)

        out = self.fin_out(db4)
        return out

## UNET

In [ ]:
import torch
import torch.nn as nn

class DiscriminativeSubNetwork(nn.Module):
    def __init__(self,in_channels=3, out_channels=3, base_channels=64, out_features=False):
        super(DiscriminativeSubNetwork, self).__init__()
        base_width = base_channels
        self.encoder_segment = EncoderDiscriminative(in_channels, base_width)
        self.decoder_segment = DecoderDiscriminative(base_width, out_channels=out_channels)
        #self.segment_act = torch.nn.Sigmoid()
        self.out_features = out_features
    def forward(self, x):
        b1,b2,b3,b4,b5,b6 = self.encoder_segment(x)
        output_segment = self.decoder_segment(b1,b2,b3,b4,b5,b6)
        if self.out_features:
            return output_segment, b2, b3, b4, b5, b6
        else:
            return output_segment
        
    def getName(self):
        return self.__class__.__name__

class EncoderDiscriminative(nn.Module):
    def __init__(self, in_channels, base_width):
        super(EncoderDiscriminative, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels,base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True))
        self.mp1 = nn.Sequential(nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(base_width,base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True))
        self.mp2 = nn.Sequential(nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(base_width*2,base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True))
        self.mp3 = nn.Sequential(nn.MaxPool2d(2))
        self.block4 = nn.Sequential(
            nn.Conv2d(base_width*4,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))
        self.mp4 = nn.Sequential(nn.MaxPool2d(2))
        self.block5 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))

        self.mp5 = nn.Sequential(nn.MaxPool2d(2))
        self.block6 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))


    def forward(self, x):
        b1 = self.block1(x)
        mp1 = self.mp1(b1)
        b2 = self.block2(mp1)
        mp2 = self.mp3(b2)
        b3 = self.block3(mp2)
        mp3 = self.mp3(b3)
        b4 = self.block4(mp3)
        mp4 = self.mp4(b4)
        b5 = self.block5(mp4)
        mp5 = self.mp5(b5)
        b6 = self.block6(mp5)
        return b1,b2,b3,b4,b5,b6

class DecoderDiscriminative(nn.Module):
    def __init__(self, base_width, out_channels=1):
        super(DecoderDiscriminative, self).__init__()

        self.up_b = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 8, base_width * 8, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 8),
                                 nn.ReLU(inplace=True))
        self.db_b = nn.Sequential(
            nn.Conv2d(base_width*(8+8), base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 8, base_width * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 8),
            nn.ReLU(inplace=True)
        )


        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 8, base_width * 4, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 4),
                                 nn.ReLU(inplace=True))
        self.db1 = nn.Sequential(
            nn.Conv2d(base_width*(4+8), base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 4, base_width * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 4),
            nn.ReLU(inplace=True)
        )

        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 4, base_width * 2, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 2),
                                 nn.ReLU(inplace=True))
        self.db2 = nn.Sequential(
            nn.Conv2d(base_width*(2+4), base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 2, base_width * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 2),
            nn.ReLU(inplace=True)
        )

        self.up3 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 2, base_width, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width),
                                 nn.ReLU(inplace=True))
        self.db3 = nn.Sequential(
            nn.Conv2d(base_width*(2+1), base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True)
        )

        self.up4 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width),
                                 nn.ReLU(inplace=True))
        self.db4 = nn.Sequential(
            nn.Conv2d(base_width*2, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True)
        )



        self.fin_out = nn.Sequential(nn.Conv2d(base_width, out_channels, kernel_size=3, padding=1))

    def forward(self, b1,b2,b3,b4,b5,b6):
        up_b = self.up_b(b6)
        cat_b = torch.cat((up_b,b5),dim=1)
        db_b = self.db_b(cat_b)

        up1 = self.up1(db_b)
        cat1 = torch.cat((up1,b4),dim=1)
        db1 = self.db1(cat1)

        up2 = self.up2(db1)
        cat2 = torch.cat((up2,b3),dim=1)
        db2 = self.db2(cat2)

        up3 = self.up3(db2)
        cat3 = torch.cat((up3,b2),dim=1)
        db3 = self.db3(cat3)

        up4 = self.up4(db3)
        cat4 = torch.cat((up4,b1),dim=1)
        db4 = self.db4(cat4)

        out = self.fin_out(db4)
        return out





# Load dei pesi

In [ ]:
WEIGHT_DIR = '/content/drive/MyDrive/Project Work CV/weights'

#AUTOENCODER
AE_DIR = '/content/drive/MyDrive/Project Work CV/paper_weights'
AE_HEADER = 'DRAEM_seg_large_ae_large_0.0001_800_bs8_'
#UNET
UNET_DIR = '/content/drive/MyDrive/Project Work CV/weights/unet'
UNET_HEADER = 'unet_weights_'

def loadWeight(model, label_name, device):
    if(model.getName() == 'ReconstructiveSubNetwork'):
        ae_file_name = AE_HEADER + label_name + '_.pckl'
        ae_load_path = os.path.join(AE_DIR,ae_file_name)
        if os.path.exists(ae_load_path):
            model.load_state_dict(torch.load(ae_load_path, map_location=device))
            print(f"AE: Pesi per la categoria '{label_name.upper()}' caricati correttamente!\n - {ae_load_path}")
        else:
            print(f"Errore: File dei pesi non trovati.\n{ae_load_path}\n")

    if(model.getName() == 'DiscriminativeSubNetwork'):
        unet_file_name = UNET_HEADER + label_name + '.pth'
        unet_load_path = os.path.join(UNET_DIR,unet_file_name)
        if os.path.exists(unet_load_path):
            model.load_state_dict(torch.load(unet_load_path, map_location=device))
            print(f"UNET: Pesi per la categoria '{label_name.upper()}' caricati correttamente!\n - {unet_load_path}")
        else:
            print(f"UNET: Errore: File dei pesi non trovati. Verifica i percorsi:\n - {unet_load_path}\n")

## Test

In [ ]:
if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')

print("using ", device)

category = 'tile'

test_dataset = MVTecTestDataset(MVTEC_DIR, category, transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model = ReconstructiveSubNetwork(in_channels=3, out_channels=3, base_width= 128)
model_seg = DiscriminativeSubNetwork(in_channels=6, out_channels=2)

loadWeight(model, category, device)
loadWeight(model_seg, category, device)
model.to(device)
model_seg.to(device)
model.eval()
model_seg.eval()
pass

In [ ]:
total_parameters = sum(p.numel() for p in model_seg.parameters())

train_parameters = sum(p.numel() for p in model_seg.parameters() if p.requires_grad)

print(f"Total parameters: {total_parameters}")
print(f"Trainable parameters: {train_parameters}")

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import numpy as np

#inizializzazione
anomaly_score_prediction = []
anomaly_score_gt = []

dim_img = 256
total_pixel_scores = np.zeros((dim_img * dim_img * len(test_dataset)))
total_gt_pixel_scores = np.zeros((dim_img * dim_img * len(test_dataset)))
mask_cnt = 0

with torch.no_grad():
    for image, mask, label in test_loader:
        image = image.to(device)

        anomaly_score_gt.append(label.item())
        true_mask_cv = mask.squeeze().numpy()

        mask = mask.to(device)

        rec_img = model(image)

        concat_img = torch.cat((rec_img, image), dim=1)

        out_mask = model_seg(concat_img)
        #probabilità maschera
        out_mask_sm = torch.softmax(out_mask, dim=1)

        out_mask_cv = out_mask_sm[0, 1, :, :].cpu().numpy()

        out_mask_averaged = torch.nn.functional.avg_pool2d(out_mask_sm[:, 1:, :, :], 21, stride=1, padding=21 // 2).cpu().numpy()
        image_score = np.max(out_mask_averaged)
        anomaly_score_prediction.append(image_score)

        flat_true_mask = true_mask_cv.flatten()
        flat_out_mask = out_mask_cv.flatten()

        total_pixel_scores[mask_cnt * dim_img * dim_img:(mask_cnt + 1) * dim_img * dim_img] = flat_out_mask
        total_gt_pixel_scores[mask_cnt * dim_img * dim_img:(mask_cnt + 1) * dim_img * dim_img] = flat_true_mask
        mask_cnt += 1


anomaly_score_prediction = np.array(anomaly_score_prediction)
anomaly_score_gt = np.array(anomaly_score_gt)


auroc_img = roc_auc_score(anomaly_score_gt, anomaly_score_prediction)


total_gt_pixel_scores = total_gt_pixel_scores.astype(np.uint8)
total_gt_pixel_scores = total_gt_pixel_scores[:dim_img * dim_img * mask_cnt]
total_pixel_scores = total_pixel_scores[:dim_img * dim_img * mask_cnt]

auroc_pixel = roc_auc_score(total_gt_pixel_scores, total_pixel_scores)
ap_pixel = average_precision_score(total_gt_pixel_scores, total_pixel_scores)

print(f"Risultati per la categoria: {category.upper()}\n")
print("Metriche: \tAUROC IMG \tAUROC PIXEL \tAVPIXEL")
print(f"Valori:  \t{auroc_img:.4f}, \t{auroc_pixel:.4f}, \t{ap_pixel:.4f}")
print(f"Percentuali:  \t{auroc_img*100}, \t{auroc_pixel*100}, \t{ap_pixel*100}")

precisions, recalls, thresholds = precision_recall_curve(total_gt_pixel_scores, total_pixel_scores)


f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-8)


optimal_idx = np.argmax(f1_scores)

optimal_f1 = f1_scores[optimal_idx]
optimal_precision = precisions[optimal_idx]
optimal_recall = recalls[optimal_idx]
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 1.0

print("Metriche: \tRECALL \tPRECISION \tF1-SCORE")
print(f"Valori: \t{optimal_recall:.4f}, \t{optimal_precision:.4f}, \t{optimal_f1:.4f}")
print(f"Percentuali: \t{optimal_recall*100}, \t{optimal_precision*100}, \t{optimal_f1*100}")

